# EDM–Fuzzy Entropy + Fault Injection (4 humidity sensors)
Pipeline sesuai diagram: baseline → fault injection per sensor → hitung EDM–Fuzzy Entropy per sensor (vector per skala) → gabung fitur → ANN + grid search.

**Catatan percepatan**: notebook ini memakai (1) downsampling & window sampling, (2) perkiraan (Monte Carlo) pada perhitungan similarity untuk menekan kompleksitas $O(N^2)$.

# Evaluasi CV utuh untuk panjang data berbeda + uji formula hidden layer (P1–P7)

Bagian ini menambahkan dua eksperimen:

## A) CV utuh pada ukuran data berbeda (contoh: 2000, 7000, 10000)
Untuk setiap ukuran `N`, ambil subsample acak dari `(X_feat, y)`, lalu jalankan **5-fold Stratified CV penuh**.

## B) Uji formula jumlah neuron hidden layer (P1–P7)
Konversi formula P1–P7 menjadi angka neuron `h`, lalu evaluasi `MLPClassifier(hidden_layer_sizes=(h,))` dengan **CV penuh** dan metrik `f1_macro`.

> Catatan: hasil CV akan tidak stabil jika jumlah window sangat sedikit atau label sangat tidak seimbang.


In [ ]:
import numpy as np, math
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

def eval_for_sizes(X, y, sizes=(2000,7000,10000), scoring="f1_macro", cv_splits=5, seed=42):
    rng=np.random.default_rng(seed)
    idx_all=np.arange(len(y))
    out={}
    for n in sizes:
        k=min(int(n), len(y))
        idx=rng.choice(idx_all, size=k, replace=False)
        Xn, yn = X[idx], y[idx]
        cv=StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
        pipe=Pipeline([("scaler", StandardScaler()),
                       ("clf", MLPClassifier(hidden_layer_sizes=(128,64), activation="tanh",
                                             max_iter=400, early_stopping=True, random_state=seed))])
        scores=cross_val_score(pipe, Xn, yn, cv=cv, scoring=scoring, n_jobs=1)
        out[n]={"mean":float(scores.mean()),"std":float(scores.std()),"scores":scores}
        print(f"N={k:5d} | mean={scores.mean():.4f} std={scores.std():.4f} | scores={np.round(scores,4)}")
    return out

def hl_formulas(I, O, Nt):
    return {
        "P1": int(2*I + 1),
        "P2": max(1, int(math.log2(max(2,I)))),
        "P3": max(1, int((I + O) / 2)),
        "P4": max(1, int((2/3) * I)),
        "P5": int(2 * I),
        "P6": max(1, int(I / 2 + 1)),
        "P7": max(1, int(0.5 * (I + O) + math.sqrt(max(1,Nt))))
    }

def eval_formulas(X, y, formulas, scoring="f1_macro", cv_splits=5, seed=42, activation="tanh"):
    cv=StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    rows=[]
    for name,h in formulas.items():
        pipe=Pipeline([("scaler", StandardScaler()),
                       ("clf", MLPClassifier(hidden_layer_sizes=(h,), activation=activation,
                                             max_iter=400, early_stopping=True, random_state=seed))])
        scores=cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=1)
        rows.append((name,h,float(scores.mean()),float(scores.std())))
        print(f"{name}: HL={h:4d} | mean={scores.mean():.4f} std={scores.std():.4f}")
    rows=sorted(rows, key=lambda t: t[2], reverse=True)
    return rows


In [ ]:
# === Global config (edit here) ===
FAST_MODE = True  # True: fastest path to get all outputs
RUN_ALL_METHODS = True  # True: compute + evaluate all entropy methods
METHOD_LIST = ["EDM-Fuzzy", "CMSE", "FME", "JSD-Fuzzy"]
DEFAULT_METHOD = "EDM-Fuzzy"  # used for plots/CM/report if you want 1 method highlighted
CACHE_FEATURES = True  # cache features per method to avoid recompute on rerun
CACHE_DIR = "cache"
EXPORT_DIR = "exports"

# FAST_MODE knobs (will override some later defaults)
FAST_MAX_PER_CLASS = 200
FAST_N_REF = 128
FAST_N_JOBS = -1
FAST_MLP_MAX_ITER = 200
FAST_CV_REPEATS = 10  # for entropy stability repeats (if used)

# Kaggle time budget guard (hours)
KAGGLE_TIME_BUDGET_H = 11.5

# Per-scenario ANN search knobs (paper table)
SCENARIO_GRID_CV = 3
SCENARIO_TEST_FRAC = 0.25
SCENARIO_MAX_PER_CLASS = 300
SCENARIO_MAX_CANDIDATES = 10
SCENARIO_MAX_ITER = 250

import os
from pathlib import Path
from IPython.display import FileLink, display

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)

def export_df(df, name, index=False):
    p_csv = Path(EXPORT_DIR) / f"{name}.csv"
    p_par = Path(EXPORT_DIR) / f"{name}.parquet"
    df.to_csv(p_csv, index=index)
    try:
        df.to_parquet(p_par, index=index)
    except Exception:
        p_par = None
    display(FileLink(str(p_csv)))
    if p_par is not None:
        display(FileLink(str(p_par)))
    return str(p_csv), (str(p_par) if p_par is not None else None)


## A) Jalankan CV untuk ukuran data berbeda
Jika `len(y) < 10000`, otomatis pakai ukuran maksimum yang tersedia.


In [ ]:
# Jalankan cell ini SETELAH X_feat dan y terbentuk.
if "X_feat" not in globals() or "y" not in globals():
    print("X_feat/y belum ada. Jalankan dulu bagian ekstraksi fitur (compute entropy -> X_feat) dan label y.")
else:
    import numpy as np
    print("X_feat shape:", X_feat.shape, " y:", y.shape, "classes:", np.unique(y))
    _ = eval_for_sizes(X_feat, y, sizes=(2000,7000,10000))


## B) Jalankan uji formula hidden layer (P1–P7)
`I` = jumlah fitur (`X_feat.shape[1]`)  
`O` = jumlah kelas (`len(unique(y))`)  
`Nt` = jumlah sampel (`len(y)`)


In [ ]:
# Jalankan cell ini SETELAH X_feat dan y terbentuk.
if "X_feat" not in globals() or "y" not in globals():
    print("X_feat/y belum ada. Jalankan dulu bagian ekstraksi fitur (compute entropy -> X_feat) dan label y.")
else:
    import numpy as np
    I = int(X_feat.shape[1])
    O = int(len(np.unique(y)))
    Nt = int(len(y))
    formulas = hl_formulas(I,O,Nt)
    print("I,O,Nt:", I,O,Nt)
    print("Formulas:", formulas)
    rows = eval_formulas(X_feat, y, formulas)
    print("\nRanked:")
    for name,h,mean,std in rows:
        print(f"{name}: HL={h:4d} | mean={mean:.4f} std={std:.4f}")


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import time, tracemalloc, logging, warnings
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# === Speed / sampling toggles (Kaggle-friendly defaults) ===
# Target: cepat di Kaggle CPU (2-4 core) tanpa meledakkan jumlah window/fitur.
USE_DOWNSAMPLE = True
USE_WINDOWING = True
USE_BALANCED_SUBSAMPLE = True
USE_RANDOM_WINDOW_SAMPLE = False  # True kalau masih terlalu lambat

# Downsample: ambil tiap DS sampel
DS = 4

# Windowing: banyak sampel, tapi ukuran window dibuat kecil agar entropy cepat
WIN = 256
STRIDE = 128

# Batasi dataset supaya entropy + gridsearch tidak lama
MAX_PER_CLASS = 200          # max window per kelas setelah label jadi
MAX_WINDOWS_TOTAL = 2000     # dipakai kalau USE_RANDOM_WINDOW_SAMPLE=True

RANDOM_SEED = 42


# Load Data (4 Sensor)

**Tujuan:** memuat time-series kelembaban, memilih **4 kolom numerik** sebagai sensor, lalu membersihkan nilai hilang.

**Input**
- File CSV `tabel_sensor4_generated.csv` (diunduh via URL GitHub).
- Terdapat kolom "kelembaban1","kelembaban2","kelembaban3","kelembaban4"

**Proses di kode**
- `load_default_data()` mengunduh CSV → `pd.read_csv(...)` → DataFrame `df`.
- `cols = ["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]`
- `X_raw = df[cols].to_numpy(dtype=float)`
- Imputasi NaN berurutan: `ffill → bfill → median` per kolom.

**Output**
- `cols`: daftar nama 4 kolom sensor yang dipakai.
- `X`: `numpy.ndarray` bentuk `(T, 4)` berisi sinyal sensor yang sudah bebas NaN.


## 1) Load data (4 sensor)
File: `tabel_sensor4_generated.csv` (diasumsikan berisi 4 kolom sensor kelembaban).

In [ ]:
import pandas as pd
import requests

file_path = "tabel_sensor4_generated.csv"

def load_default_data():
    url = "https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/tabel_sensor4_generated.csv"
    response = requests.get(url)
    response.raise_for_status()
    from io import StringIO
    return pd.read_csv(StringIO(response.text))


df = load_default_data()
df.head(), df.shape


In [ ]:
# pilih 4 kolom
cols = ["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]
X_raw = df[cols].to_numpy(dtype=float)

# imputasi NaN sederhana (ffill → bfill → median)
X_df = pd.DataFrame(X_raw, columns=cols)
X_df = X_df.ffill().bfill().fillna(X_df.median(numeric_only=True))

# guard NaN (jelas kalau masih ada)
if X_df.isna().any().any():
    raise ValueError("Error-nya jelas: fitur X masih ada NaN setelah imputasi. Periksa data input.")

X = X_df.to_numpy()

# optional downsample (Kaggle speed)
if 'USE_DOWNSAMPLE' in globals() and USE_DOWNSAMPLE:
    X_ds = X[::DS]
else:
    X_ds = X

print("Columns:", cols)
print("Shape X:", X.shape, "Shape X_ds:", X_ds.shape)


In [ ]:
# --- Window params (pakai yg sudah didefinisikan di atas) ---
if "WIN" not in globals(): WIN = 256
if "STRIDE" not in globals(): STRIDE = max(1, WIN//2)
# Windowing parameters (FIX: aman walau X_ds belum didefinisikan)
# Catatan: cell ini sebaiknya berada SETELAH pembuatan X_ds. Kalau belum ada, kita fallback ke X.
import numpy as np


if "X_ds" not in globals():
    if "X" in globals():
        X_ds = X
    else:
        raise NameError("X_ds belum ada dan X belum ada. Jalankan dulu cell load/preprocess data.")

N = X_ds.shape[0]
if WIN > N:
    WIN = max(128, 2**int(np.floor(np.log2(max(128, N//2)))))
STRIDE = min(STRIDE, max(1, WIN//2))
print("WIN/STRIDE:", WIN, STRIDE, "N:", N)


In [ ]:
# Sanity check: jumlah window harus cukup banyak untuk klasifikasi
total_windows = None
if 'W' in globals(): total_windows = getattr(W, "shape", [None])[0]
if total_windows is not None and total_windows < 200:
    print("WARNING: total_windows terlalu sedikit:", total_windows)
    print("Saran cepat: coba WIN=512 STRIDE=128 atau WIN=1024 STRIDE=256 (atau turunkan STRIDE).")


# Fault Injection (per Sensor)

**Tujuan:** membangkitkan data *faulty* dari baseline dengan menyisipkan fault **per sensor** sesuai skenario.

## A. Simulator fault 1D (per sensor)
Semua simulator menerima **sinyal 1D** `x` bentuk `(T',)` dan menghasilkan:
- `y`: sinyal setelah fault `(T',)`
- `m`: mask boolean lokasi fault `(T',)`

- `simulate_drift_fault`: menambah drift linier `t*intensity`.
- `simulate_spike_fault`: menambah spike periodik (besar spike ∝ `std(x)`).
- `simulate_bias_fault`: menambah offset konstan (bias).
- `simulate_hardware_fault`: kombinasi **stuck** (nilai diganti nilai lain) dan **loss** (NaN).

## B. Multi-fault dalam satu sensor
`simulate_multiple_faults(x, faults)` menerapkan daftar fault berurutan; mask digabung `OR` sehingga menandai titik yang terkena salah satu fault.

## C. Injeksi ke 4 sensor
`inject_faults_multisensor(X, scenario_faults)`:
- loop ke-4 sensor: injeksi skenario yang sama, tetapi seed berbeda per sensor
- menangani NaN (loss) dengan `ffill → bfill → median`

**Output:**  
- `Y`: `(T',4)` data setelah fault  
- `M`: `(T',4)` mask fault per sensor

## D. Label fault per window
`window_fault_label(M, WIN, STRIDE, thr)`:
- untuk tiap window: hitung rasio fault per sensor = `mean(mask_window)`
- window dianggap fault bila **ada sensor** dengan rasio > `thr`

**Output:** `is_fault_win` boolean per window.


## 3) Fault injection (sesuai skenario)
Fault disisipkan **per sensor**. Skenario minimal 6 kombinasi 2-fault.

Output: dataset windows berlabel (kelas) + mask fault untuk menentukan window yang 'faulty'.

In [ ]:

# --- Fault simulators (subtler, different seed per call) ---
def simulate_drift_fault(x, intensity=0.02, seed=None):
    rng=np.random.default_rng(seed)
    alpha = intensity
    drift = np.arange(len(x)) * alpha
    y=x+drift; m=np.abs(drift)>1e-6; return y,m

def simulate_spike_fault(x, intensity=0.08, p=0.015, seed=None):
    rng=np.random.default_rng(seed)
    tau = max(1, int(1.0 / p)) if p > 0 else len(x)
    spikes = (np.arange(len(x)) % tau == 0).astype(float) * (intensity * np.nanstd(x))
    y=x+spikes; m=spikes!=0; return y,m

def simulate_bias_fault(x, bias=0.08, seed=None):
    y=x+bias; m=np.ones(len(x),bool); return y,m

def simulate_hardware_fault(x, stuck_prob=0.08, loss_prob=0.05, seed=None):
    rng=np.random.default_rng(seed)
    n = len(x)
    rand_vals = rng.random(n)
    idx=rng.integers(n, size=n)
    m1 = rand_vals < stuck_prob
    y = x.copy()
    y[m1] = x[idx[m1]]
    m2 = rand_vals < loss_prob
    y[m2] = np.nan
    return y, (m1 | m2)

def simulate_multiple_faults(x, faults, seed=None):
    y=x.copy(); m=np.zeros(len(x),bool)
    for f,kw in faults:
        y,mi=f(y,**kw,seed=seed); m|=mi
    return y,m


def simulate_choose_one(x, options, seed=None):
    rng = np.random.default_rng(seed)
    f, kw = options[rng.integers(len(options))]
    return f(x, **kw, seed=seed)

# fault scenario ditambah 2,3,4 fault
SCENARIOS = {
    # 1
    "faulty": [(
        simulate_choose_one, {
            "options": [
                (simulate_drift_fault, {"intensity":0.02}),
                (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                (simulate_bias_fault, {"bias":0.08}),
                (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05}),
            ]
        }
    )],

    # 2
    "drift": [(simulate_drift_fault, {"intensity":0.02})],
    "spike": [(simulate_spike_fault, {"intensity":0.08, "p":0.015})],
    "bias": [(simulate_bias_fault, {"bias":0.08})],
    "hardware": [(simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],

    # 3
    "bias+malfunc": [(simulate_bias_fault, {"bias":0.08}),
                     (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "spike+malfunc": [(simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                      (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "spike+bias": [(simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                   (simulate_bias_fault, {"bias":0.08})],
    "drift+malfunc": [(simulate_drift_fault, {"intensity":0.02}),
                      (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "drift+bias": [(simulate_drift_fault, {"intensity":0.02}),
                   (simulate_bias_fault, {"bias":0.08})],
    "drift+spike": [(simulate_drift_fault, {"intensity":0.02}),
                    (simulate_spike_fault, {"intensity":0.08, "p":0.015})],

    # 4
    "spike+bias+malfunc": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "drift+bias+malfunc": [
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "spike+drift+malfunc": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "drift+spike+bias": [
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
    ],

    # 5
    "spike+bias+malfunc+drift": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05}),
        (simulate_drift_fault, {"intensity":0.02}),
    ],
}


In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

def make_windows(X, win, stride):
    Xn = np.asarray(X, dtype=np.float32)
    N = Xn.shape[0]
    if win <= 0 or stride <= 0:
        raise ValueError("win dan stride harus > 0")
    if N < win:
        return np.empty((0, win, Xn.shape[1]), dtype=np.float32), np.array([], dtype=int)
    view = sliding_window_view(Xn, window_shape=win, axis=0)  # (N-win+1, win, F)
    starts = np.arange(0, N - win + 1, stride, dtype=int)
    return view[starts], starts


In [ ]:

def inject_faults_multisensor(X, scenario_faults, seed=0):
    # X: (T,4) -> Y: (T,4), M: (T,4) boolean mask
    rng = np.random.default_rng(seed)
    Y = X.copy()
    M = np.zeros_like(Y, dtype=bool)
    for s in range(Y.shape[1]):
        y, m = simulate_multiple_faults(Y[:,s], scenario_faults, seed=int(rng.integers(1e9)))
        Y[:,s] = y
        M[:,s] = m
    # replace NaN (from malfunc loss) with forward fill then median per sensor
    Ydf = pd.DataFrame(Y)
    Ydf = Ydf.ffill().bfill().fillna(Ydf.median(numeric_only=True))
    return Ydf.to_numpy(), M

def window_fault_label(mask, win, stride, fault_ratio_thr=0.02):
    # mask: (T,4) -> per window label faulty if any sensor has >thr fraction
    T = len(mask)
    if win > T:
        return np.zeros(0, dtype=bool), np.array([], dtype=int)
    from numpy.lib.stride_tricks import sliding_window_view
    Wm = sliding_window_view(mask, window_shape=win, axis=0)[::stride]  # (Nwin, win, 4)
    ratio = Wm.mean(axis=1)  # (Nwin, 4)
    y = (ratio > fault_ratio_thr).any(axis=1)
    starts = np.arange(0, T-win+1, stride, dtype=int)
    return y, starts


# Build windowed dataset across scenarios + normal
fault_ratio_thr = 0.02
datasets = []
labels = []
scenario_names = ["normal"] + list(SCENARIOS.keys())

# simpan sinyal time-series per skenario (untuk analisis CV berbasis panjang data)
series_by_scenario = {}


# normal
W0, starts0 = make_windows(X_ds, WIN, STRIDE)
datasets.append(W0); labels.append(np.zeros(len(W0), dtype=int))
series_by_scenario["normal"] = X_ds

# scenarios
for k, (name, faults) in enumerate(SCENARIOS.items(), start=1):
    Y, M = inject_faults_multisensor(X_ds, faults, seed=100+k)
    series_by_scenario[name] = Y
    is_fault_win, starts = window_fault_label(M, WIN, STRIDE, fault_ratio_thr=fault_ratio_thr)
    Wk, _ = make_windows(Y, WIN, STRIDE)
    # label all windows as scenario k (including potentially non-faulty parts),
    # but you can also filter to only faulty windows:
    Wk = Wk[is_fault_win]
    datasets.append(Wk); labels.append(np.full(len(Wk), k, dtype=int))
    print(name, "windows:", len(Wk))

W_all = np.concatenate(datasets, axis=0)
y_all = np.concatenate(labels, axis=0)
print("Total windows:", W_all.shape, "Classes:", np.unique(y_all, return_counts=True))

# Pastikan format window = (N, WIN, 4). Jika terbalik (N,4,WIN) maka transpose.
if W_all.ndim==3 and W_all.shape[1]==4 and W_all.shape[2]==WIN:
    W_all = W_all.transpose(0,2,1)



### Optional: balanced sampling per class (reduction)
Jika data besar, batasi `max_per_class` untuk mempercepat perhitungan entropy dan ANN.

In [ ]:
def balanced_subsample(Xw, y, max_per_class=600, seed=0):
    rng=np.random.default_rng(seed)
    keep=[]
    for c in np.unique(y):
        idx=np.where(y==c)[0]
        if len(idx)>max_per_class:
            idx=rng.choice(idx, size=max_per_class, replace=False)
        keep.append(idx)
    keep=np.concatenate(keep)
    rng.shuffle(keep)
    return Xw[keep], y[keep]

def random_window_subsample(Xw, y, max_total=4000, seed=0):
    if len(Xw) <= max_total:
        return Xw, y
    rng=np.random.default_rng(seed)
    idx=rng.choice(np.arange(len(Xw)), size=max_total, replace=False)
    return Xw[idx], y[idx]

W_s, y_s = W_all, y_all

if USE_BALANCED_SUBSAMPLE:
    W_s, y_s = balanced_subsample(W_s, y_s, max_per_class=MAX_PER_CLASS, seed=RANDOM_SEED)

if USE_RANDOM_WINDOW_SAMPLE:
    W_s, y_s = random_window_subsample(W_s, y_s, max_total=MAX_WINDOWS_TOTAL, seed=RANDOM_SEED)

# Guard shape untuk menghindari IndexError 2D vs 3D
if W_s.ndim != 3:
    raise ValueError(
        f"Error: W_s harus 3D (N, WIN, S). Saat ini shape={W_s.shape}. "
        "Pastikan make_windows() dipanggil dan USE_WINDOWING=True."
    )

print("After subsample toggles:", W_s.shape, np.unique(y_s, return_counts=True))


# EDM–Fuzzy Entropy (Fitur Utama) + Metode Pembanding

**Tujuan:** untuk setiap window & setiap sensor, menghitung **vektor entropy multi-skala**
dengan beberapa metode untuk perbandingan: **EDM–Fuzzy, CMSE, FME, dan JSD–Fuzzy**.

## Langkah per skala `s` (EDM–Fuzzy)
1) **Coarse-graining** (`coarse_grain_mean`):  
   - Input: `x` (window 1D), `s`  
   - Output: `y` yang diperkasar (panjang ~ `WIN/s`) dengan rata-rata blok.

2) **Embedding** (`embed_matrix`):  
   - Input: `y`, dimensi `m`  
   - Output: matriks `V_m` bentuk `(Nemb, m)` (sliding window).

3) **Jarak Euclidean & Similarity fuzzy** (`fuzzy_phi`):  
   - Menghitung `phi_m` sebagai rata-rata similarity fuzzy:  
     
     \[
     \mu(d)=
rac{1}{1+(d/r)^2}
     \]
   - **Percepatan:** sampling `n_ref` embedding sebagai referensi (Monte Carlo),
     sehingga tidak perlu semua pasangan (mengurangi beban O(N²)).

4) **Entropy** (`edm_fuzzy_entropy_1d`):  
   - Hitung `phi_m` dan `phi_{m+1}` →  
     \[
     E(s)=\ln\left(
rac{\phi_m}{\phi_{m+1}}
ight)
     \]

**Output:** untuk satu sensor pada satu window → vektor `E` bentuk `(S,)`.

Catatan: untuk metode lain (CMSE, FME, JSD–Fuzzy), alur tetap mengikuti skala `s`
dengan definisi entropy masing-masing.


## 4) Entropy Multi-Method (EDM–Fuzzy, CMSE, FME, JSD–Fuzzy)
Implementasi ringkas mengikuti blok diagram:
- EDM–Fuzzy: seperti definisi di atas (jarak Euclidean + similarity fuzzy).
- CMSE: Composite Multiscale Sample Entropy (SampEn).
- FME: Fuzzy Multiscale Entropy (jarak Chebyshev + similarity fuzzy).
- JSD–Fuzzy: Jensen–Shannon divergence antar distribusi similarity fuzzy (m vs m+1).

**Percepatan:** gunakan sampling indeks embedding (Monte Carlo) sehingga tidak perlu semua pasangan vektor.

**Catatan:** pipeline di bawah kini menghitung semua metode **secara paralel** untuk perbandingan.


### 🔧 Peningkatan akurasi JSD–Fuzzy Entropy

Versi lama `jsd_fuzzy_entropy_1d` hanya mengeluarkan **1 fitur/skala**: nilai JSD
(divergensi *bentuk* histogram similarity fuzzy untuk embedding `m` vs `m+1`).
JSD membuang informasi **magnitude/lokasi** distribusi similarity, padahal itu
diskriminatif untuk klasifikasi fault.

Versi `rich=True` (default sekarang) mengeluarkan **4 fitur/skala**:
`[jsd, fe, mean_m, std_m]` — dengan `fe = log(mean(mu_m)/mean(mu_m1))`
(fuzzy-entropy ala EDM). Sampel similarity sudah dihitung untuk JSD, jadi
penambahan ini hampir tanpa biaya komputasi.

**Hasil (17-kelas fault, ANN MLP 128–64, rata-rata 3 split):**

| Varian JSD–Fuzzy | #fitur | akurasi |
|---|---|---|
| lama (1 fitur/skala) | 40 | ~0.341 |
| rich, n_ref=64, bins=20 | 160 | ~0.390 |
| rich, n_ref=128, bins=40 | 160 | ~0.424 |

Ablasi: menambahkan `mean_m` memberi lonjakan terbesar (`jsd`→`jsd+mean`:
0.34→0.41); `fe`+`std` menambah sedikit lagi. Untuk hasil terbaik, naikkan juga
`n_ref` (64→128) dan `jsd_bins` (20→40) pada cell konfigurasi entropy.
Set `rich=False` untuk perilaku lama.

In [ ]:
S = 10  # jumlah skala (akan dipakai untuk reshape & feature dim)
def coarse_grain_mean(x, s):
    n = (len(x)//s)*s
    if n <= 0:
        return np.array([], dtype=float)
    xs = x[:n].reshape(-1, s).mean(axis=1)
    return xs

def coarse_grain_multi(x, s):
    # Composite coarse-graining dengan offset 0..s-1 (CMSE)
    ys = []
    for k in range(s):
        n = (len(x)-k)//s
        if n <= 0:
            continue
        y = x[k:k+n*s].reshape(-1, s).mean(axis=1)
        if len(y) > 0:
            ys.append(y)
    return ys

def embed_matrix(y, m):
    # y: (L,) -> (L-m+1, m)
    L = len(y)
    if L < m:
        return np.empty((0, m), dtype=float)
    return np.lib.stride_tricks.sliding_window_view(y, m)

def fuzzy_phi(V, r, n_ref=256, seed=0):
    # V: (N,m). Approximate phi by sampling reference vectors i
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.nan
    if N > n_ref:
        ref = rng.choice(N, size=n_ref, replace=False)
    else:
        ref = np.arange(N)
    A = V[ref]
    a2 = np.sum(A*A, axis=1, keepdims=True)
    b2 = np.sum(V*V, axis=1, keepdims=True).T
    d2 = np.maximum(a2 + b2 - 2*(A @ V.T), 0.0)
    rr = r*r + 1e-24
    mu = 1.0/(1.0 + d2/rr)          # no sqrt
    mu[np.arange(len(ref)), ref] = 0.0  # no python loop
    Bi = mu.sum(axis=1) / (N-1)
    return Bi.mean()


def fuzzy_phi_cheb(V, r, n_ref=256, seed=0):
    # Similarity fuzzy dengan jarak Chebyshev (FME)
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.nan
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    A = V[ref]
    d = np.max(np.abs(A[:, None, :] - V[None, :, :]), axis=2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    for ri, i in enumerate(ref):
        mu[ri, i] = 0.0
    Bi = mu.sum(axis=1) / (N-1)
    return Bi.mean()

def sample_entropy_1d(y, m, r):
    # SampEn dasar untuk CMSE (pakai jarak Chebyshev)
    V_m  = embed_matrix(y, m)
    V_m1 = embed_matrix(y, m+1)

    def _count_similar(V):
        N = V.shape[0]
        if N < 2:
            return 0, 0
        count = 0
        total = 0
        for i in range(N-1):
            d = np.max(np.abs(V[i+1:] - V[i]), axis=1)
            count += np.sum(d <= r)
            total += (N - i - 1)
        return count, total

    c_m, t_m = _count_similar(V_m)
    c_m1, t_m1 = _count_similar(V_m1)
    if t_m == 0 or t_m1 == 0 or c_m == 0 or c_m1 == 0:
        return np.nan
    return -np.log((c_m1 / t_m1) / (c_m / t_m))

def edm_fuzzy_entropy_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0):
    # returns entropy vector [len(scales)]
    out=[]
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        phi_m  = fuzzy_phi(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        phi_m1 = fuzzy_phi(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (phi_m is None) or (phi_m1 is None) or (phi_m<=0) or (phi_m1<=0) or np.isnan(phi_m) or np.isnan(phi_m1):
            out.append(np.nan)
        else:
            out.append(np.log(phi_m/phi_m1))
    return np.array(out, dtype=float)

def cmse_1d(x, scales, m=2, r_ratio=0.2):
    # CMSE: Composite Multiscale Sample Entropy
    out = []
    for s in scales:
        ys = coarse_grain_multi(x, s)
        if not ys:
            out.append(np.nan); continue
        ent_list = []
        for y in ys:
            if len(y) < (m+2):
                continue
            r = r_ratio * np.std(y, ddof=1)
            ent_list.append(sample_entropy_1d(y, m=m, r=r))
        if len(ent_list) == 0:
            out.append(np.nan)
        else:
            out.append(np.nanmean(ent_list))
    return np.array(out, dtype=float)

def fme_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0):
    # FME: Fuzzy Multiscale Entropy (jarak Chebyshev)
    out=[]
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        phi_m  = fuzzy_phi_cheb(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        phi_m1 = fuzzy_phi_cheb(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (phi_m is None) or (phi_m1 is None) or (phi_m<=0) or (phi_m1<=0) or np.isnan(phi_m) or np.isnan(phi_m1):
            out.append(np.nan)
        else:
            out.append(np.log(phi_m/phi_m1))
    return np.array(out, dtype=float)

def fuzzy_similarity_samples(V, r, n_ref=256, seed=0):
    # Ambil sampel nilai similarity fuzzy (untuk JSD)
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.array([], dtype=float)
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    A = V[ref]
    a2 = np.sum(A*A, axis=1, keepdims=True)
    b2 = np.sum(V*V, axis=1, keepdims=True).T
    d2 = a2 + b2 - 2*(A @ V.T)
    d2 = np.maximum(d2, 0.0)
    d = np.sqrt(d2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    for ri, i in enumerate(ref):
        mu[ri, i] = np.nan
    return mu[~np.isnan(mu)].ravel()

def jsd_fuzzy_entropy_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0, bins=20, rich=True):
    # JSD-Fuzzy (IMPROVED): selain JSD shape-divergence antara distribusi similarity
    # fuzzy (m vs m+1), kita tambahkan deskriptor MAGNITUDE distribusi yang sebelumnya
    # dibuang oleh JSD (yang hanya menangkap perbedaan *bentuk*).
    # Per skala -> [jsd, fe, mean_m, std_m] bila rich=True (default), atau [jsd] bila rich=False.
    #   jsd    : Jensen-Shannon divergence histogram similarity m vs m+1 (seperti versi lama)
    #   fe     : fuzzy-entropy = log(mean_mu_m / mean_mu_m1)  -> level kesamaan absolut
    #   mean_m : rata-rata similarity (lokasi distribusi)
    #   std_m  : sebaran similarity (skala distribusi)
    # Sampel similarity sudah dihitung utk JSD, jadi tambahan fitur ini hampir gratis
    # namun menaikkan akurasi klasifikasi ANN secara signifikan (~0.34 -> ~0.42 pada
    # eksperimen 17-kelas fault). Set rich=False untuk perilaku lama (1 fitur/skala).
    out=[]
    per = 4 if rich else 1
    bin_edges = np.linspace(0.0, 1.0, bins+1)
    eps = 1e-12
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.extend([np.nan]*per); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        mu_m  = fuzzy_similarity_samples(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        mu_m1 = fuzzy_similarity_samples(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (len(mu_m) == 0) or (len(mu_m1) == 0):
            out.extend([np.nan]*per); continue
        p,_ = np.histogram(mu_m,  bins=bin_edges)
        q,_ = np.histogram(mu_m1, bins=bin_edges)
        p = p.astype(float); q = q.astype(float)
        if p.sum() == 0 or q.sum() == 0:
            out.extend([np.nan]*per); continue
        p /= p.sum(); q /= q.sum()
        m_ = 0.5 * (p + q)
        kl_p = np.sum(p * np.log((p + eps) / (m_ + eps)))
        kl_q = np.sum(q * np.log((q + eps) / (m_ + eps)))
        jsd = 0.5 * (kl_p + kl_q)
        if rich:
            fe = np.log((mu_m.mean() + eps) / (mu_m1.mean() + eps))
            out.extend([jsd, fe, mu_m.mean(), mu_m.std()])
        else:
            out.append(jsd)
    return np.array(out, dtype=float)

# quick sanity check on one window/sensor
scales = np.arange(1, S+1)
e = edm_fuzzy_entropy_1d(W_s[0,:,0], scales=scales, m=2, r_ratio=0.2, n_ref=256, seed=0)
e


In [ ]:
# Guard: pastikan scale tidak bikin coarse-grain kosong (menghindari 'Mean of empty slice')
m_embed = 2  # samakan dengan m pada entropy
min_len = m_embed + 2
max_scale = max(1, WIN // min_len)

# scales bisa datang dari cell sebelumnya; pastikan list int
scales = [int(s) for s in scales]
scales = [s for s in scales if s <= max_scale]
if len(scales) == 0:
    scales = [1]

# update S supaya selalu konsisten dengan scales yang benar-benar dipakai
S = len(scales)
scales = np.array(scales, dtype=int)

print("Using scales:", scales.tolist(), "S:", S, "max_scale:", max_scale)


# Fitur Akhir: Konkatenasi 4 Sensor (Multi-Method)

`compute_features_entropy(W, scales, method=...)`

**Input**
- `W`: `(Nwin, WIN, 4)` kumpulan window
- `scales`: `1..S`
- `method`: `EDM-Fuzzy`, `CMSE`, `FME`, `JSD-Fuzzy`

**Proses**
- untuk tiap window:
  - hitung entropy `(S,)` untuk sensor 1..4
  - gabungkan menjadi `(4*S,)`: `[E1, E2, E3, E4]`

**Output**
- `F_by_method`: dict fitur `(Nwin, 4*S)` untuk setiap metode
- NaN (jika ada) diimputasi dengan median per kolom.


## 5) Hitung entropy per sensor → gabung fitur
Entropy dihitung per sensor menghasilkan vector:
- Sensor1: $E_1(1..S)$
- Sensor2: $E_2(1..S)$
- Sensor3: $E_3(1..S)$
- Sensor4: $E_4(1..S)$

Fitur akhir = konkatenasi $[E_1,E_2,E_3,E_4]$ ukuran `4*S`.

Catatan A: uji beberapa pilihan skala untuk CV (kestabilan entropy).
Catatan B: semua metode dihitung **bersamaan** agar perbandingan mudah.


In [ ]:
from joblib import Parallel, delayed

START_TIME = time.time()

def time_left_sec():
    if 'KAGGLE_TIME_BUDGET_H' not in globals():
        return None
    return KAGGLE_TIME_BUDGET_H * 3600 - (time.time() - START_TIME)

def ensure_window_3d(W, name="W"):
    if W.ndim != 3:
        raise ValueError(f"{name} harus 3D (N, WIN, S). Dapat {W.shape}.")
    return W

def sanitize_features(F, name="F"):
    Fdf = pd.DataFrame(F)
    if Fdf.isna().any().any():
        warnings.warn(f"{name} masih ada NaN; imputasi median diterapkan.")
        Fdf = Fdf.fillna(Fdf.median(numeric_only=True))
    if Fdf.isna().any().any():
        raise ValueError(f"Error-nya jelas: fitur {name} masih ada NaN setelah imputasi.")
    return Fdf.to_numpy()

def compute_features_entropy(W, scales, method=None, m=2, r_ratio=0.2, n_ref=256, jsd_bins=20, seed=0, n_jobs=-1, prefer="processes"):
    # W: (Nwin, win, 4) -> (Nwin, 4*len(scales))
    W = ensure_window_3d(W, name="W")
    Nwin, win, ns = W.shape
    if method is None:
        method = DEFAULT_METHOD
    method_key = str(method).strip().lower()

    def entropy_1d(x, seed_local):
        if method_key == 'edm-fuzzy':
            return edm_fuzzy_entropy_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local)
        if method_key == 'cmse':
            return cmse_1d(x, scales=scales, m=m, r_ratio=r_ratio)
        if method_key == 'fme':
            return fme_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local)
        if method_key == 'jsd-fuzzy':
            return jsd_fuzzy_entropy_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local, bins=jsd_bins)
        raise ValueError("Unknown method: %s" % method)

    def one_window(i):
        feats = [entropy_1d(W[i,:,s], seed_local=seed+1000*i+19*s) for s in range(ns)]
        return np.concatenate(feats, axis=0)

    if n_jobs == 1 or Nwin <= 1:
        F = np.vstack([one_window(i) for i in range(Nwin)])
    else:
        F = Parallel(n_jobs=n_jobs, prefer=prefer)(delayed(one_window)(i) for i in range(Nwin))
        F = np.vstack(F)
    return F


# Utility logging untuk footprint komputasi
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

def run_with_metrics(label, fn):
    # Catatan: footprint = estimasi peak memory Python via tracemalloc
    tracemalloc.start()
    t0 = time.perf_counter()
    c0 = time.process_time()
    result = fn()
    t1 = time.perf_counter()
    c1 = time.process_time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    metrics = {
        "wall_s": t1 - t0,
        "cpu_s": c1 - c0,
        "peak_mem_mb": peak / (1024*1024)
    }
    logging.info("%s | wall=%.2fs cpu=%.2fs peak_mem=%.2f MB", label, metrics["wall_s"], metrics["cpu_s"], metrics["peak_mem_mb"])
    return result, metrics

# === Entropy params (speed) ===
# S kecil + n_ref kecil = jauh lebih cepat, masih cukup untuk baseline eksperimen.
S = 10
scales = np.arange(1, S+1)
m = 2
r_ratio = 0.2
n_ref = 128
jsd_bins = 40

# Default: fokus ke EDM-Fuzzy dulu (metode lain bisa ditambah lagi kalau perlu)
methods = METHOD_LIST if RUN_ALL_METHODS else [DEFAULT_METHOD]
F_by_method = {}
F_metrics = {}

# PENTING: cache key HARUS mencakup konfigurasi fitur (jumlah skala, n_ref,
# jsd_bins, rich). Bug sebelumnya: cache hanya di-key oleh nama metode, jadi
# saat config/kode diubah (mis. JSD versi 'rich') fitur lama yang basi tetap
# dimuat dan tak pernah dihitung ulang -> JSD muncul 24 fitur (versi lama
# rich=False) alih-alih 4 fitur/skala. Tag config bikin cache invalid otomatis
# begitu salah satu parameter berubah.
JSD_RICH = True
CFG_TAG = f"S{len(scales)}_m{m}_r{r_ratio}_nref{n_ref}_bins{jsd_bins}_rich{int(JSD_RICH)}"

for name in methods:
    safe = name.replace("-", "_").replace(" ", "_")
    cache_path = os.path.join(CACHE_DIR, f"F_{safe}__{CFG_TAG}.npy")
    if CACHE_FEATURES and os.path.exists(cache_path):
        F_by_method[name] = sanitize_features(np.load(cache_path), name=f"F_{name}")
        print(name, "feature shape (cache):", F_by_method[name].shape, "| cfg:", CFG_TAG)
        continue
    Fm, mtr = run_with_metrics(
        "Entropy %s" % name,
        lambda n=name: compute_features_entropy(W_s, scales=scales, method=n, m=m, r_ratio=r_ratio, n_ref=n_ref, jsd_bins=jsd_bins, seed=7)
    )
    F_by_method[name] = sanitize_features(Fm, name=f"F_{name}")
    F_metrics[name] = mtr
    print(name, "feature shape:", F_by_method[name].shape, "| cfg:", CFG_TAG)
    # simpan cache per-metode DENGAN tag config, di dalam loop -> semua metode tersimpan
    # (sebelumnya hanya metode terakhir yang ter-cache karena di luar loop).
    if CACHE_FEATURES:
        np.save(cache_path, F_by_method[name])

pd.DataFrame(F_metrics).T

# Default: gunakan EDM–Fuzzy agar blok berikut tetap kompatibel
F = F_by_method[DEFAULT_METHOD]
if CACHE_FEATURES:
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(os.path.join(CACHE_DIR, "y_windows.npy"), y_s)
    np.save(os.path.join(CACHE_DIR, "F_default.npy"), F)
    print("Saved per-method caches (cfg:", CFG_TAG, ") + y_windows.npy")


In [ ]:
# Alias kompatibilitas (beberapa blok lama masih pakai X_feat/y)
X_feat = F
y = y_s
print('Alias set: X_feat=F', getattr(X_feat,'shape',None), 'y', getattr(y,'shape',None))


In [ ]:
# Display computing footprint per entropy method
feature_metrics_table = pd.DataFrame(F_metrics).T
feature_metrics_table


# CV (Stabilitas Entropy)

`entropy_cv_report(F, S)`

**Input**
- `F`: `(Nwin, 4*S)` fitur entropy gabungan

**Proses**
- reshape menjadi `E = F.reshape(Nwin, 4, S)`
- hitung mean & std di axis window (Nwin)
- CV per sensor per skala: `std / |mean|`
- `cv_avg`: rata-rata CV dari 4 sensor pada tiap skala

**Output**
- `cv_by_sensor`: `(4, S)`
- `cv_avg`: `(S,)` untuk plot stabilitas vs skala (semakin kecil → semakin stabil)


## 6) CV (Coefficient of Variation) untuk menunjukkan kestabilan entropy
CV dihitung **per skala** dari sebaran entropy (mis. across windows). Ini memudahkan menunjukkan apakah entropy stabil terhadap variasi sampel.

Catatan A: jika ingin menguji CV untuk beberapa pilihan `S`, jalankan loop pada beberapa `S`.
Catatan B: pada versi ini, CV dihitung **untuk semua metode** agar mudah dibandingkan.


In [ ]:
import numpy as np
import pandas as pd
# === CV stabilitas entropy vs panjang data (tanpa downsampling & tanpa windowing) ===

CV_DATA_LENGTHS = [2000, 7000, 10000]   # panjang data (jumlah sampel mentah) yang ingin dibandingkan
CV_N_REPEATS = 30                      # jumlah pengulangan sampling segmen per skenario per panjang
CV_RANDOM_SEED = 123

rng_cv = np.random.default_rng(CV_RANDOM_SEED)

def entropy_cv_report(F, S=None, nsensors=4):
    # ROBUST: jumlah fitur per sensor di-infer dari F (mendukung JSD-Fuzzy "rich"
    # yang menghasilkan 4 fitur/skala, sehingga kolom = nsensors * (4*S), bukan nsensors*S).
    N = F.shape[0]
    if F.shape[1] % nsensors != 0:
        raise ValueError(f"F.shape[1]={F.shape[1]} tidak habis dibagi nsensors={nsensors}")
    S_feat = F.shape[1] // nsensors
    E = F.reshape(N, nsensors, S_feat)
    mean = E.mean(axis=0)
    std  = E.std(axis=0, ddof=0)
    cv   = std / (np.abs(mean)+1e-12)
    return cv, cv.mean(axis=0)

def sample_segments(X_ts, seg_len, n_repeats, rng):
    # X_ts: (T,4) -> W: (n_repeats, seg_len, 4)
    T = X_ts.shape[0]
    if seg_len > T:
        raise ValueError(f"seg_len {seg_len} > T {T}")
    if seg_len == T:
        return X_ts[None, :, :]
    starts = rng.integers(0, T-seg_len+1, size=n_repeats)
    return np.stack([X_ts[s:s+seg_len] for s in starts], axis=0)

# hitung CV untuk tiap skenario x panjang data x metode
rows = []

if "series_by_scenario" not in globals():
    raise RuntimeError("series_by_scenario tidak ditemukan. Pastikan cell pembentukan skenario sudah dijalankan.")

for scen_name, X_ts in series_by_scenario.items():
    for L in CV_DATA_LENGTHS:
        if L > X_ts.shape[0]:
            continue
        Wcv = sample_segments(X_ts, seg_len=L, n_repeats=CV_N_REPEATS, rng=rng_cv)
        for method_name in F_by_method.keys():
            Fcv = compute_features_entropy(Wcv, scales=scales, method=method_name,
                                          m=m, r_ratio=r_ratio, n_ref=n_ref,
                                          jsd_bins=jsd_bins, seed=CV_RANDOM_SEED)
            _, cv_avg = entropy_cv_report(Fcv, S=S)
            for s_idx, cv_val in enumerate(cv_avg, start=1):
                rows.append({
                    "scenario": scen_name,
                    "data_length": int(L),
                    "scale": int(s_idx),
                    "method": method_name,
                    "cv": float(cv_val),
                    "n_repeats": int(Wcv.shape[0])
                })

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

cv_table = pd.DataFrame(rows).sort_values(["scenario","data_length","method","scale"]).reset_index(drop=True)
cv_table


In [ ]:
# Recompute mean entropy per sensor (robust)
N = F.shape[0]
NSENSORS = W_s.shape[2] if 'W_s' in globals() else 4
S_feat = F.shape[1]//NSENSORS
assert F.shape[1] % NSENSORS == 0, f"F harus punya {NSENSORS}*S kolom"
E = F.reshape(N, NSENSORS, S_feat)
meanE = E.mean(axis=0)

mean_entropy_table = pd.DataFrame(
    meanE,
    index=[f"Sensor {i}" for i in range(1, NSENSORS+1)],
    columns=[f"s={i}" for i in range(1, S_feat+1)]
)
mean_entropy_table


### Visualisasi entropy rata-rata per skala (opsional)
Menampilkan mean entropy untuk melihat trend vs skala.

In [ ]:
# Recompute mean entropy per sensor (robust)
N = F.shape[0]
NSENSORS = W_s.shape[2] if 'W_s' in globals() else 4
S_feat = F.shape[1]//NSENSORS
assert F.shape[1] % NSENSORS == 0, f"F harus punya {NSENSORS}*S kolom"
E = F.reshape(N, NSENSORS, S_feat)
meanE = E.mean(axis=0)

mean_entropy_table = pd.DataFrame(
    meanE,
    index=[f"Sensor {i}" for i in range(1, NSENSORS+1)],
    columns=[f"s={i}" for i in range(1, S_feat+1)]
)
mean_entropy_table


# Klasifikasi Fault (ANN + Grid Search)

**Input**
- `F`: `(N, 4*S)` fitur entropy
- `y_s`: label kelas (0=normal, 1..K skenario fault)

## Split data
- Default: `train_test_split(..., stratify=y)` agar proporsi kelas seimbang.
- Opsi: `USE_TIME_SPLIT=True` memakai `time_split` untuk mengurangi leakage akibat overlap window.

## Model
- Pipeline: `StandardScaler()` → `MLPClassifier()`
- GridSearch menyapu: `hidden_layer_sizes`, `alpha`, `learning_rate_init`, `activation`.
- Evaluasi: classification report + confusion matrix.

**Output**
- `best`: estimator terbaik hasil `GridSearchCV`
- metrik: accuracy & macro-F1 pada test set
- baseline pembanding: Logistic Regression dan Random Forest.

Catatan: untuk perbandingan metode, cukup ganti `F` dari `F_by_method["EDM-Fuzzy"]` ke
`F_by_method["CMSE"]`, `F_by_method["FME"]`, atau `F_by_method["JSD-Fuzzy"]`.


## 7) Fault detection / classification dengan ANN
Input = gabungan vector entropy (4×S). Output = kelas fault (1..n) + normal (0).

Arsitektur hidden layer mengikuti best practice (Heaton):
- $H_1$ antara input dan output, sering dipakai: $\lfloor\frac{2}{3}I + O\rfloor$
- Alternatif: $H_1=I$, $H_2=\lfloor I/2 \rfloor$

Di sini dibuat beberapa kandidat dan dilakukan grid search.

In [ ]:

def eval_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    return {
        "model": name,
        "accuracy": float(accuracy_score(yte, pred)),
        "macro_f1": float(f1_score(yte, pred, average="macro"))
    }


In [ ]:

# Optional: gunakan split berbasis urutan (mengurangi leakage akibat overlap window)
USE_TIME_SPLIT = False  # set True untuk split berdasarkan urutan window

def time_split(F, y, test_frac=0.25):
    n=len(F); cut=int(np.floor((1-test_frac)*n))
    return F[:cut], F[cut:], y[:cut], y[cut:]


# Upgrade evaluasi & model (untuk naikkan performa saat EDM-Fuzzy rendah)

Perubahan utama di notebook ini:

1) **Metrik**: pakai `f1_macro` (lebih adil saat kelas tidak seimbang).
2) **Cek imbalance**: tampilkan distribusi label & baseline skor.
3) **Baseline models**: Logistic Regression, SVM-RBF, RandomForest (sering lebih kuat untuk fitur tabular seperti entropy).
4) **ANN tuning**: grid arsitektur fleksibel + `early_stopping=True`, tapi **GridSearchCV `n_jobs=1`** (hindari double-parallelism).
5) **Stabilisasi fitur**: rekomendasi parameter EDM-Fuzzy yang lebih “stabil” (mis. `n_ref` lebih kecil & `S` tidak terlalu besar).

> Catatan: kalau baseline (mis. SVM) juga mentok ~0.3, biasanya fitur EDM-Fuzzy-nya yang belum cocok (window/stride/threshold/imbalance), bukan ANN-nya.


In [ ]:
from collections import Counter
import numpy as np

_X = globals().get("X_feat", None)
_y = globals().get("y", None)

print("X_feat:", getattr(_X, "shape", None))
print("y:", getattr(_y, "shape", None))
if _y is not None:
    ctr = Counter(_y.tolist() if hasattr(_y, "tolist") else list(_y))
    print("class counts:", dict(sorted(ctr.items())))
else:
    print("y belum tersedia di notebook (cek cell fitur & label).")


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring="f1_macro"

models = {
    "LogReg(balanced)": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "SVM-RBF(balanced)": SVC(kernel="rbf", class_weight="balanced"),
    "RF(balanced)": RandomForestClassifier(n_estimators=400, random_state=42, class_weight="balanced_subsample", n_jobs=-1),
}

for name, clf in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)]) if not isinstance(clf, RandomForestClassifier) else clf
    scores = cross_val_score(pipe, X_feat, y, cv=cv, scoring=scoring, n_jobs=1)
    print(f"{name:18s} mean={scores.mean():.4f} std={scores.std():.4f}  scores={np.round(scores,4)}")


In [ ]:
# ANN (MLP) evaluasi dengan metrik f1_macro + arsitektur fleksibel
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Guard NaN
if np.isnan(X_feat).any():
    raise ValueError("Error-nya jelas: fitur X masih ada NaN, jadi semua fit di GridSearch gagal karena MLPClassifier tidak bisa menerima NaN.")

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        random_state=42,
        max_iter=400,
        early_stopping=True,
        n_iter_no_change=15
    ))
])

# Kandidat arsitektur (HL2 tidak dipaksa = 1/2 HL1)
candidates = [
    (32,), (64,), (128,), (256,),
    (64,16), (64,32), (64,64),
    (128,32), (128,64), (128,128),
    (256,64), (256,128),
    (128,64,32)
]

param_grid = {
    "mlp__hidden_layer_sizes": candidates,
    "mlp__activation": ["relu", "tanh"],
    "mlp__alpha": [1e-5, 1e-4, 1e-3],
    "mlp__learning_rate_init": [1e-3, 3e-4],
    "mlp__batch_size": [64, 128]
}

gs = GridSearchCV(
    pipe, param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=1,      # penting: hindari double-parallelism
    verbose=2
)
gs.fit(X_feat, y)
print("BEST:", gs.best_params_)
print("BEST f1_macro:", gs.best_score_)


In [ ]:
# Train/test split + GridSearch ANN (lebih cepat di Kaggle)
compare_methods = RUN_ALL_METHODS  # True=bandingkan metode; False=EDM-Fuzzy saja
train_metrics_by_method = {}

def build_hidden_candidates(I, O):
    base = sorted(set([
        max(8, I//4),
        max(16, I//2),
        max(32, int(np.floor((2/3)*I + O))),
        I,
        min(2*I, 512),
    ]))
    cand=[(h,) for h in base]
    for h1 in base:
        for h2 in sorted(set([max(4, h1//4), max(8, h1//2)])):
            cand.append((h1, h2))
    for h1 in base[-2:]:
        cand.append((h1, max(8, h1//2), max(4, h1//4)))
    cand=list(dict.fromkeys(cand))[:20]
    return cand

def train_ann_for_F(F_local, label_name):
    Xtr, Xte, ytr, yte = time_split(F_local, y_s, test_frac=0.25) if USE_TIME_SPLIT else train_test_split(
        F_local, y_s, test_size=0.25, random_state=42, stratify=y_s
    )
    I = F_local.shape[1]
    O = len(np.unique(y_s))
    candidates = build_hidden_candidates(I, O)

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(max_iter=400, random_state=42, early_stopping=True, n_iter_no_change=10))
    ])
    param_grid = {
        "mlp__hidden_layer_sizes": candidates,
        "mlp__alpha": [1e-4, 1e-3, 1e-2],
        "mlp__learning_rate_init": [1e-3, 3e-4],
        "mlp__activation": ["relu", "tanh"],
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    gs = GridSearchCV(pipe, param_grid=param_grid, cv=cv, n_jobs=1, scoring="accuracy", verbose=1)
    gs.fit(Xtr, ytr)

    best = gs.best_estimator_
    pred = best.predict(Xte)
    metrics = {
        "best_params": gs.best_params_,
        "best_cv_acc": float(gs.best_score_),
        "test_acc": float(accuracy_score(yte, pred)),
        "macro_f1": float(f1_score(yte, pred, average="macro"))
    }
    return {"gs": gs, "Xtr": Xtr, "Xte": Xte, "ytr": ytr, "yte": yte, "pred": pred, "metrics": metrics, "label": label_name}

if compare_methods:
    train_results = {}
    for name in methods:
        res, mtr = run_with_metrics(f"ANN {name}", lambda n=name: train_ann_for_F(F_by_method[n], n))
        train_results[name] = res
        train_metrics_by_method[name] = {**res["metrics"], **mtr}
    pd.DataFrame(train_metrics_by_method).T
    base_res = train_results[DEFAULT_METHOD]
else:
    edm_res, mtr = run_with_metrics(f"ANN {DEFAULT_METHOD}", lambda: train_ann_for_F(F, DEFAULT_METHOD))
    train_metrics_by_method[DEFAULT_METHOD] = {**edm_res["metrics"], **mtr}

gs = base_res["gs"]
Xtr, Xte, ytr, yte = base_res["Xtr"], base_res["Xte"], base_res["ytr"], base_res["yte"]
pred = base_res["pred"]


In [ ]:
# Display computing footprint per method during ANN training
train_metrics_table = pd.DataFrame(train_metrics_by_method).T
train_metrics_table


In [ ]:
# Combined comparison table (feature + training metrics)
combined_metrics_table = pd.concat(
    [feature_metrics_table.add_prefix('feat_'), train_metrics_table.add_prefix('train_')],
    axis=1
)
combined_metrics_table


In [ ]:
# Quick stats preview for combined metrics
combined_metrics_table.describe()


In [ ]:

best = gs.best_estimator_
pred = best.predict(Xte)

print(classification_report(yte, pred, labels=np.arange(len(scenario_names)), target_names=scenario_names))
print('Accuracy:', accuracy_score(yte, pred))
print('Macro-F1:', f1_score(yte, pred, average="macro"))
cm = confusion_matrix(yte, pred, labels=np.arange(len(scenario_names)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=scenario_names)
disp.plot(xticks_rotation=45)
plt.show()


## Arsitektur ANN yang menghasilkan Precision–Recall–F1 Score

Skor **Precision/Recall/F1** di atas berasal dari **model ANN terbaik** hasil *Grid Search* (`gs.best_estimator_`), yaitu pipeline:

`StandardScaler → MLPClassifier`

Arsitekturnya dapat dituliskan sebagai:

- **Jumlah node input (I)** = jumlah fitur pada vektor entropy `F` (kolom pada `Xtr`)
- **Hidden layer** = `hidden_layer_sizes` dari MLP terbaik (bisa 1 atau 2 hidden layer sesuai kandidat grid)
- **Jumlah node output (O)** = jumlah kelas fault (`len(mlp.classes_)`)

Di bawah ini ditampilkan ringkasan arsitektur + visualisasi skematik dan grafik Precision/Recall/F1 per kelas.


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support

best = gs.best_estimator_
mlp = best.named_steps["mlp"]

I = Xtr.shape[1]
hidden = mlp.hidden_layer_sizes if isinstance(mlp.hidden_layer_sizes, tuple) else (mlp.hidden_layer_sizes,)
O = len(mlp.classes_)

print("=== Arsitektur ANN terbaik (hasil Grid Search) ===")
print(f"Input layer : {I} node")
print(f"Hidden layer: {hidden} (jumlah layer = {len(hidden)})")
print(f"Output layer: {O} node (kelas = {list(mlp.classes_)})")
print("Activation  :", mlp.activation)
print("Alpha (L2)  :", mlp.alpha)
print("LR init     :", mlp.learning_rate_init)

# --- Visualisasi skema arsitektur (ringkas) ---
def plot_mlp_schema(I, hidden, O):
    layers = [("Input", I)] + [(f"Hidden {k+1}", h) for k,h in enumerate(hidden)] + [("Output", O)]
    n = len(layers)
    fig, ax = plt.subplots(figsize=(min(12, 2.2*n), 2.8))
    ax.axis("off")
    xs = list(range(n))
    for i,(name,size) in enumerate(layers):
        ax.add_patch(plt.Rectangle((xs[i]-0.45, -0.35), 0.9, 0.7, fill=False))
        ax.text(xs[i], 0.0, f"{name}\n({size} node)", ha="center", va="center")
        if i < n-1:
            ax.annotate("", xy=(xs[i+1]-0.55, 0), xytext=(xs[i]+0.55, 0), arrowprops=dict(arrowstyle="->"))
    ax.set_xlim(-0.8, n-0.2)
    ax.set_ylim(-1, 1)
    plt.show()

plot_mlp_schema(I, hidden, O)

# --- Visualisasi Precision / Recall / F1 per kelas ---
rep = classification_report(yte, pred, labels=np.arange(len(scenario_names)), target_names=scenario_names, output_dict=True, zero_division=0)
classes = [c for c in scenario_names if c in rep]
prec = [rep[c]["precision"] for c in classes]
rec  = [rep[c]["recall"] for c in classes]
f1   = [rep[c]["f1-score"] for c in classes]

x = np.arange(len(classes))
w = 0.25
plt.figure(figsize=(max(10, 0.6*len(classes)), 4))
plt.bar(x - w, prec, width=w, label="Precision")
plt.bar(x,     rec,  width=w, label="Recall")
plt.bar(x + w, f1,   width=w, label="F1")
plt.xticks(x, classes, rotation=45, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Precision / Recall / F1 per Kelas (Test Set)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Baseline comparisons (trained on same split as ANN)
baseline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, multi_class="auto"))
])

baseline_rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

results = []
# ANN best estimator
results.append({
    "model":"ANN (GridSearch best)",
    "accuracy": float(accuracy_score(yte, pred)),
    "macro_f1": float(f1_score(yte, pred, average="macro"))
})
results.append(eval_model("LogReg", baseline_lr, Xtr, ytr, Xte, yte))
results.append(eval_model("RandomForest", baseline_rf, Xtr, ytr, Xte, yte))

pd.DataFrame(results).sort_values(["macro_f1","accuracy"], ascending=False)


In [ ]:

# === Per-scenario ANN architecture search (accuracy vs compute cost) ===

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold

# Helpers

def count_mlp_params(I, hidden, O):
    sizes = [I] + list(hidden) + [O]
    return int(sum((sizes[i] + 1) * sizes[i+1] for i in range(len(sizes)-1)))

def balanced_binary_subsample(X, y, max_per_class=300, seed=0):
    rng = np.random.default_rng(seed)
    keep = []
    for c in [0, 1]:
        idx = np.where(y == c)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        keep.append(idx)
    keep = np.concatenate(keep)
    rng.shuffle(keep)
    return X[keep], y[keep]

def small_arch_candidates(I, O, max_cand=10):
    base = sorted(set([
        max(8, I//4),
        max(16, I//2),
        max(32, int(np.floor((2/3)*I + O))),
        I,
        min(2*I, 256),
    ]))
    cand = [(h,) for h in base]
    for h1 in base[:3]:
        cand.append((h1, max(8, h1//2)))
    cand = list(dict.fromkeys(cand))[:max_cand]
    return cand

def fit_best_mlp(X, y, seed=42, max_iter=250, cv=3):
    # Impute + scale always, to avoid NaN errors
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            random_state=seed,
            max_iter=max_iter,
            early_stopping=True,
            n_iter_no_change=10
        ))
    ])
    I = X.shape[1]
    O = len(np.unique(y))
    candidates = small_arch_candidates(I, O, max_cand=SCENARIO_MAX_CANDIDATES)
    param_grid = {
        "mlp__hidden_layer_sizes": candidates,
        "mlp__alpha": [1e-4, 1e-3],
        "mlp__learning_rate_init": [1e-3],
        "mlp__activation": ["relu", "tanh"],
    }
    cv = StratifiedKFold(n_splits=cv, shuffle=True, random_state=seed)
    gs = GridSearchCV(pipe, param_grid=param_grid, cv=cv, n_jobs=1, scoring="f1_macro", verbose=0)
    t0 = time.perf_counter()
    gs.fit(X, y)
    t1 = time.perf_counter()
    return gs, (t1 - t0)

# Build per-scenario datasets (binary: normal vs scenario)
scenario_results = []

if "F" not in globals() or "y_s" not in globals():
    print("F/y_s belum tersedia. Jalankan cell fitur terlebih dahulu.")
else:
    for k, name in enumerate(scenario_names):
        if name == "normal":
            continue
        if time_left_sec() is not None and time_left_sec() < 600:
            print("Time budget hampir habis, menghentikan per-scenario search.")
            break

        mask = (y_s == 0) | (y_s == k)
        X_bin = F[mask]
        y_bin = (y_s[mask] == k).astype(int)

        # Balanced subsample for speed + class balance
        X_bin, y_bin = balanced_binary_subsample(X_bin, y_bin, max_per_class=SCENARIO_MAX_PER_CLASS, seed=RANDOM_SEED + k)

        # Train/test split
        Xtr, Xte, ytr, yte = train_test_split(
            X_bin, y_bin, test_size=SCENARIO_TEST_FRAC, random_state=RANDOM_SEED, stratify=y_bin
        )

        # Grid search (best architecture)
        gs, wall_s = fit_best_mlp(Xtr, ytr, seed=RANDOM_SEED, max_iter=SCENARIO_MAX_ITER, cv=SCENARIO_GRID_CV)
        best = gs.best_estimator_
        pred = best.predict(Xte)
        f1 = f1_score(yte, pred, average="macro")
        acc = accuracy_score(yte, pred)

        mlp = best.named_steps["mlp"]
        hidden = mlp.hidden_layer_sizes if isinstance(mlp.hidden_layer_sizes, tuple) else (mlp.hidden_layer_sizes,)
        I = Xtr.shape[1]
        O = len(np.unique(y_bin))
        params = count_mlp_params(I, hidden, O)

        # Inference time proxy (per 1000 samples)
        t0 = time.perf_counter()
        _ = best.predict(Xte[:min(1000, len(Xte))])
        t1 = time.perf_counter()
        inf_ms_per_sample = (t1 - t0) * 1000.0 / max(1, min(1000, len(Xte)))

        scenario_results.append({
            "scenario": name,
            "n_train": int(len(Xtr)),
            "n_test": int(len(Xte)),
            "best_hidden": str(hidden),
            "activation": mlp.activation,
            "alpha": float(mlp.alpha),
            "lr_init": float(mlp.learning_rate_init),
            "cv_f1": float(gs.best_score_),
            "test_f1": float(f1),
            "test_acc": float(acc),
            "params": params,
            "train_wall_s": float(wall_s),
            "inf_ms_per_sample": float(inf_ms_per_sample),
        })

scenario_table = pd.DataFrame(scenario_results)
scenario_table


# Eksperimen Sensitivitas Jumlah Skala `S`

`run_experiment_S(...)`

**Tujuan:** melihat trade-off:
- Stabilitas entropy (CV) vs
- Performa klasifikasi (Accuracy/Macro-F1)
ketika `S` diubah.

**Proses per nilai S**
1) Hitung fitur `F` dengan ukuran `4*S`
2) Hitung `CV_mean` (rata-rata cv_avg)
3) Train MLP sederhana (tanpa grid besar) untuk estimasi performa
4) Simpan ringkasan ke tabel `exp`

**Output**
- `exp`: DataFrame berisi kolom `S, CV_mean, test_acc, macro_f1, I, H1`
- Plot CV vs S, Accuracy vs S, Macro-F1 vs S


## 8) Eksperimen: variasi jumlah skala `S` (untuk CV + akurasi)
Bagian ini mengulang pipeline fitur untuk beberapa `S` agar terlihat hubungan kestabilan (CV) dan performa klasifikasi.

Untuk speed: turunkan `MAX_PER_CLASS`, `n_ref`, atau `S_list`.

In [ ]:
def run_experiment_S(W, y, S_list=(5,8,10,12), m=2, r_ratio=0.2, n_ref=128, seed=7, use_time_split=False, method="EDM-Fuzzy"):
    rows=[]
    for S in S_list:
        scales=np.arange(1,S+1)
        if method is None:
            method = DEFAULT_METHOD
        F = compute_features_entropy(W, scales=scales, method=method, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed)
        Fdf=pd.DataFrame(F).fillna(pd.DataFrame(F).median(numeric_only=True))
        F=Fdf.to_numpy()
        cv_by_sensor, cv_avg = entropy_cv_report(F, S=S)

        # simple MLP (no big grid) for quick trend
        Xtr, Xte, ytr, yte = time_split(F, y, test_frac=0.25) if use_time_split else train_test_split(F, y, test_size=0.25, random_state=42, stratify=y)
        I = F.shape[1]
        O = len(np.unique(y))
        H1 = int(np.floor((2/3)*I + O))

        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=(H1,), max_iter=200, random_state=42))
        ])
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        rows.append({
            "S": S,
            "CV_mean": float(np.mean(cv_avg)),
            "test_acc": float(accuracy_score(yte, pred)),
            "macro_f1": float(f1_score(yte, pred, average="macro")),
            "I": I,
            "H1": H1,
            "method": method
        })

    exp = pd.DataFrame(rows)
    return exp


In [ ]:

# === Robust experiment computation (fully self-contained, no NameError) ===

if "S_list" not in globals():
    S_list = (5, 8, 10, 12)

if "exp" not in globals():
    exp = run_experiment_S(
        W_s, y_s,
        S_list=S_list,
        m=2,
        r_ratio=0.2,
        n_ref=128,
        seed=7,
        use_time_split=USE_TIME_SPLIT
    )

# ---- Plotcs ----
plt.figure()
plt.plot(exp["S"], exp["CV_mean"], marker="o")
plt.xlabel("S (max scale)")
plt.ylabel("Mean CV (entropy)")
plt.title("Entropy stability vs S")
plt.show()

plt.figure()
plt.plot(exp["S"], exp["test_acc"], marker="o")
plt.xlabel("S (max scale)")
plt.ylabel("Test accuracy")
plt.title("ANN accuracy vs S")
plt.show()

plt.figure()
plt.plot(exp["S"], exp["macro_f1"], marker="o")
plt.xlabel("S (max scale)")
plt.ylabel("Macro-F1")
plt.title("Macro-F1 vs S")
plt.show()


In [ ]:

plt.figure()
plt.plot(exp["S"], exp["macro_f1"], marker='o')
plt.xlabel("S (max scale)")
plt.ylabel("Macro-F1")
plt.title("Macro-F1 vs S")
plt.show()


# Lima Skenario Klasifikasi Paper (S1–S5)

Sesuai rencana paper, evaluasi dilakukan pada **5 skenario klasifikasi** berikut:

| Skenario | Deskripsi | Kelas |
|---|---|---|
| S1 | Fault-free vs All-fault (binary) | 2 kelas |
| S2 | Fault-free vs Each Single-fault (5-class) | 5 kelas: normal, drift, spike, bias, hardware |
| S3 | Fault-free vs Multi-fault (beberapa kombinasi 2-fault) | multi kelas |
| S4 | Single-fault identification (4-class tanpa normal) | 4 kelas |
| S5 | All-class multiclass (normal + semua tipe fault) | banyak kelas |

Setiap skenario dievaluasi untuk **EDM-Fuzzy** dan **JSD-Fuzzy** dengan ANN-LM (MLPClassifier) + Grid Search yang identik.

In [ ]:
# === Definisi Lima Skenario Klasifikasi Paper (S1–S5) ===
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import warnings

# Pastikan variabel global tersedia
assert "y_s" in globals(), "Jalankan cell pembentukan dataset (fault injection + balanced subsample) terlebih dahulu."
assert "F_by_method" in globals() and len(F_by_method) >= 2, "Jalankan cell komputasi fitur entropy terlebih dahulu."

METHODS_PAPER = ["EDM-Fuzzy", "JSD-Fuzzy"]

# --- Bangun label per skenario ---
# Scenario 1 (S1): binary – normal (0) vs faulty (1)
def build_labels_S1(y_full):
    return (y_full > 0).astype(int)

# Scenario 2 (S2): fault-free vs 4 single-fault types (5-class)
# single-fault indices di y_s: lihat scenario_names
SINGLE_FAULT_NAMES = ["drift", "spike", "bias", "hardware"]
def build_labels_S2(y_full, scenario_names):
    sf_idx = [scenario_names.index(n) for n in SINGLE_FAULT_NAMES if n in scenario_names]
    keep = np.where((y_full == 0) | np.isin(y_full, sf_idx))[0]
    y_raw = y_full[keep]
    # Remap: 0 tetap 0, tiap sf_idx -> 1,2,3,4
    remap = {0: 0}
    for new_label, old_idx in enumerate(sf_idx, start=1):
        remap[old_idx] = new_label
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

# Scenario 3 (S3): fault-free vs 2-fault combinations (multi kelas)
MULTI2_FAULT_NAMES = ["bias+malfunc", "spike+malfunc", "spike+bias",
                       "drift+malfunc", "drift+bias", "drift+spike"]
def build_labels_S3(y_full, scenario_names):
    mf2_idx = [scenario_names.index(n) for n in MULTI2_FAULT_NAMES if n in scenario_names]
    keep = np.where((y_full == 0) | np.isin(y_full, mf2_idx))[0]
    y_raw = y_full[keep]
    remap = {0: 0}
    for new_label, old_idx in enumerate(mf2_idx, start=1):
        remap[old_idx] = new_label
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

# Scenario 4 (S4): single-fault identification (4-class, tanpa normal)
def build_labels_S4(y_full, scenario_names):
    sf_idx = [scenario_names.index(n) for n in SINGLE_FAULT_NAMES if n in scenario_names]
    keep = np.where(np.isin(y_full, sf_idx))[0]
    y_raw = y_full[keep]
    remap = {old: new for new, old in enumerate(sf_idx)}
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

# Scenario 5 (S5): all-class – normal + semua single-fault + 2-fault combinations
def build_labels_S5(y_full, scenario_names):
    all_named = SINGLE_FAULT_NAMES + MULTI2_FAULT_NAMES
    all_idx = [scenario_names.index(n) for n in all_named if n in scenario_names]
    keep = np.where((y_full == 0) | np.isin(y_full, all_idx))[0]
    y_raw = y_full[keep]
    all_classes = sorted(set([0] + all_idx))
    remap = {old: new for new, old in enumerate(all_classes)}
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

print("Skenario S1–S5 siap dibentuk.")
print("scenario_names:", scenario_names[:10], "...")


## RQ1 – Validasi Feature Matrix JSD-Fuzzy vs EDM-Fuzzy

Memastikan JSD-Fuzzy menghasilkan feature matrix yang valid: dimensi sama, bebas NaN/Inf, dan layak menjadi input classifier.

In [ ]:
# === RQ1: Validasi Feature Matrix (EDM-Fuzzy vs JSD-Fuzzy) ===

rq1_rows = []
for method in METHODS_PAPER:
    if method not in F_by_method:
        print(f"SKIP: {method} tidak ada di F_by_method.")
        continue
    F_m = F_by_method[method]
    nan_count = int(np.isnan(F_m).sum())
    inf_count = int(np.isinf(F_m).sum())
    rq1_rows.append({
        "Method": method,
        "n_samples": F_m.shape[0],
        "n_scale_features": F_m.shape[1],
        "NaN_values": nan_count,
        "Infinite_values": inf_count,
        "Status": "Valid" if (nan_count == 0 and inf_count == 0) else "INVALID"
    })

rq1_table = pd.DataFrame(rq1_rows)
print("\n=== RQ1: Feature Matrix Validation ===")
print(rq1_table.to_string(index=False))

# Export
rq1_table.to_csv("exports/rq1_feature_matrix_validation.csv", index=False)
print("\n[Saved] exports/rq1_feature_matrix_validation.csv")


## RQ2 – Stabilitas Fitur Entropy (Coefficient of Variation)

Membandingkan CV per kelas dan per skala antara EDM-Fuzzy dan JSD-Fuzzy.
- **CV = std / |mean|** per skala per kelas
- **CV Reduction (%)** = ((CV_EDM - CV_JSD) / CV_EDM) × 100


In [ ]:
# === RQ2: CV Stability per Class per Scale ===
import warnings

def compute_cv_per_class_scale(F_m, y_labels, n_scales_paper=10, nsensors=4):
    """
    Compute CV per kelas per skala.
    Hanya ambil n_scales_paper scale pertama per sensor,
    lalu rata-rata across sensors (karena paper fokus pada S=1..10).
    """
    n_feat = F_m.shape[1]
    # JSD-Fuzzy rich menghasilkan 4 fitur/skala, EDM-Fuzzy 1 fitur/skala
    feat_per_sensor = n_feat // nsensors
    feat_per_scale = feat_per_sensor // n_scales_paper  # 1 (EDM) atau 4 (JSD rich)
    if feat_per_scale < 1:
        feat_per_scale = 1

    rows = []
    for cls in np.unique(y_labels):
        idx = np.where(y_labels == cls)[0]
        Fc = F_m[idx]
        for s in range(1, n_scales_paper + 1):
            # Ambil kolom yang sesuai untuk scale s (averaged across sensors, take first feature)
            scale_vals = []
            for sensor in range(nsensors):
                col_start = sensor * feat_per_sensor + (s - 1) * feat_per_scale
                col_end = col_start + feat_per_scale
                if col_end > n_feat:
                    continue
                vals = Fc[:, col_start:col_end].mean(axis=1)  # mean over features in this scale
                scale_vals.append(vals)
            if len(scale_vals) == 0:
                continue
            vals_all = np.concatenate(scale_vals)
            mu = np.abs(vals_all.mean())
            sigma = vals_all.std(ddof=0)
            cv = sigma / (mu + 1e-12)
            rows.append({"class_idx": int(cls), "scale": s, "mean": float(mu),
                         "std": float(sigma), "cv": float(cv)})
    return pd.DataFrame(rows)

print("Menghitung CV per kelas per skala untuk EDM-Fuzzy dan JSD-Fuzzy...")

rq2_all = {}
for method in METHODS_PAPER:
    if method not in F_by_method:
        continue
    df_cv = compute_cv_per_class_scale(F_by_method[method], y_s)
    df_cv["method"] = method
    rq2_all[method] = df_cv

rq2_combined = pd.concat(rq2_all.values(), ignore_index=True)

# Summary: mean CV per method per class
rq2_summary = (rq2_combined.groupby(["method", "class_idx"])["cv"]
               .mean().reset_index()
               .rename(columns={"cv": "mean_CV"}))

# CV pivot: method x scale (averaged across all classes)
rq2_by_scale = (rq2_combined.groupby(["method", "scale"])["cv"]
                .mean().reset_index().pivot(index="scale", columns="method", values="cv"))
rq2_by_scale.columns.name = None

# CV Reduction per scale
if "EDM-Fuzzy" in rq2_by_scale.columns and "JSD-Fuzzy" in rq2_by_scale.columns:
    rq2_by_scale["CV_Reduction_%"] = (
        (rq2_by_scale["EDM-Fuzzy"] - rq2_by_scale["JSD-Fuzzy"]) /
        (rq2_by_scale["EDM-Fuzzy"] + 1e-12) * 100
    )

print("\n=== RQ2: Mean CV per Scale (semua kelas) ===")
print(rq2_by_scale.round(4).to_string())

# Mean CV global
for method in METHODS_PAPER:
    if method in rq2_all:
        mean_cv = rq2_all[method]["cv"].mean()
        print(f"{method} | Global Mean CV = {mean_cv:.4f}")

# Export
rq2_by_scale.to_csv("exports/rq2_cv_per_scale.csv")
rq2_combined.to_csv("exports/rq2_cv_per_class_scale.csv", index=False)
print("\n[Saved] exports/rq2_cv_per_scale.csv, rq2_cv_per_class_scale.csv")


In [ ]:
# === RQ2: Plot CV per Scale (EDM-Fuzzy vs JSD-Fuzzy) ===
scales_plot = rq2_by_scale.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: CV vs Scale
ax = axes[0]
for method in METHODS_PAPER:
    if method in rq2_by_scale.columns:
        ax.plot(scales_plot, rq2_by_scale[method], marker="o", label=method)
ax.set_xlabel("Scale Factor s")
ax.set_ylabel("Mean CV (all classes)")
ax.set_title("RQ2: Entropy Stability (CV) vs Scale")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: CV Reduction per scale
ax2 = axes[1]
if "CV_Reduction_%" in rq2_by_scale.columns:
    colors = ["green" if v >= 0 else "red" for v in rq2_by_scale["CV_Reduction_%"]]
    ax2.bar(scales_plot, rq2_by_scale["CV_Reduction_%"], color=colors, alpha=0.7)
    ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax2.set_xlabel("Scale Factor s")
    ax2.set_ylabel("CV Reduction (%) [EDM→JSD]")
    ax2.set_title("RQ2: CV Reduction per Scale")
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("exports/rq2_cv_stability_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] exports/rq2_cv_stability_plot.png")


## RQ3 – Feature Separability (Boxplot Mean Entropy per Kelas)

Membandingkan distribusi **mean entropy across scales** per kelas fault antara EDM-Fuzzy dan JSD-Fuzzy.

**Mean Entropy = (s1 + s2 + ... + s10) / 10** (menggunakan fitur skala pertama per sensor)


In [ ]:
# === RQ3: Boxplot Feature Separability ===
import matplotlib.pyplot as plt
import numpy as np

def compute_mean_entropy_per_sample(F_m, n_scales_paper=10, nsensors=4, feat_offset=0):
    """
    Hitung mean entropy per sampel (rata-rata s1..s10, sensor pertama, fitur pertama).
    feat_offset: untuk JSD-Fuzzy rich, ambil fitur ke-0 (jsd) dari 4 fitur per skala.
    """
    n_feat = F_m.shape[1]
    feat_per_sensor = n_feat // nsensors
    feat_per_scale = feat_per_sensor // n_scales_paper
    if feat_per_scale < 1:
        feat_per_scale = 1

    # Ambil satu fitur representatif per scale, averaged across sensors
    scale_cols = []
    for s in range(n_scales_paper):
        sensor_vals = []
        for sensor in range(nsensors):
            col = sensor * feat_per_sensor + s * feat_per_scale + min(feat_offset, feat_per_scale - 1)
            if col < n_feat:
                sensor_vals.append(F_m[:, col])
        if len(sensor_vals) > 0:
            scale_cols.append(np.mean(sensor_vals, axis=0))

    if len(scale_cols) == 0:
        return np.zeros(F_m.shape[0])
    return np.mean(scale_cols, axis=0)

# Gunakan S2 label (5-class: normal + 4 single fault) sebagai contoh utama
# Bangun index dan label S2
keep_s2, y_s2 = build_labels_S2(y_s, scenario_names)
s2_class_names = ["normal"] + SINGLE_FAULT_NAMES

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax_idx, method in enumerate(METHODS_PAPER):
    if method not in F_by_method:
        continue
    ax = axes[ax_idx]
    F_m = F_by_method[method]
    F_sub = F_m[keep_s2]
    mean_ent = compute_mean_entropy_per_sample(F_sub)

    data_by_class = [mean_ent[y_s2 == c] for c in sorted(np.unique(y_s2))]
    class_labels = [s2_class_names[c] if c < len(s2_class_names) else f"class {c}"
                    for c in sorted(np.unique(y_s2))]

    bp = ax.boxplot(data_by_class, tick_labels=class_labels, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    colors_box = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]
    for patch, color in zip(bp["boxes"], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f"RQ3: {method}\n(Scenario S2 – 5 kelas)")
    ax.set_xlabel("Fault Class")
    ax.set_ylabel("Mean Entropy (s1–s10)")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("exports/rq3_boxplot_separability_S2.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] exports/rq3_boxplot_separability_S2.png")


In [ ]:
# === RQ3: Tabel Ringkasan Separability (Median per Kelas) ===
rq3_rows = []
keep_s2, y_s2 = build_labels_S2(y_s, scenario_names)
s2_class_names_local = ["normal"] + SINGLE_FAULT_NAMES

for method in METHODS_PAPER:
    if method not in F_by_method:
        continue
    F_m = F_by_method[method]
    F_sub = F_m[keep_s2]
    mean_ent = compute_mean_entropy_per_sample(F_sub)
    for c in sorted(np.unique(y_s2)):
        vals = mean_ent[y_s2 == c]
        cls_name = s2_class_names_local[c] if c < len(s2_class_names_local) else f"class {c}"
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        rq3_rows.append({
            "Method": method, "Class": cls_name,
            "Median": round(med, 4), "Q1": round(q1, 4), "Q3": round(q3, 4),
            "IQR": round(q3 - q1, 4), "Mean": round(vals.mean(), 4), "Std": round(vals.std(), 4)
        })

rq3_table = pd.DataFrame(rq3_rows)
print("\n=== RQ3: Feature Separability Summary (Scenario S2) ===")
print(rq3_table.to_string(index=False))
rq3_table.to_csv("exports/rq3_separability_summary.csv", index=False)
print("\n[Saved] exports/rq3_separability_summary.csv")


## RQ4 – Performa Fault Detection: Lima Skenario Klasifikasi (EDM-Fuzzy vs JSD-Fuzzy)

Setiap skenario (S1–S5) dievaluasi secara **fair**:
- Dataset, label, dan data split identik untuk kedua metode.
- Grid Search ANN-LM dijalankan **terpisah** untuk masing-masing metode (masing-masing mencari arsitektur terbaiknya sendiri).
- Metrik utama: **F1-score (macro)**, accuracy, precision, recall.


In [ ]:
# === RQ4: Helper – Fair ANN-LM Grid Search per Skenario ===
import time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report
)

RQ4_TEST_FRAC = 0.25
RQ4_RANDOM_SEED = 42
RQ4_CV_SPLITS = 3
RQ4_MAX_ITER = 300
RQ4_N_ITER_NO_CHANGE = 15

def rq4_arch_candidates(I, O, max_cand=12):
    base = sorted(set([
        max(8, I // 4), max(16, I // 2),
        max(32, int(round((2/3)*I + O))),
        I, min(2*I, 256),
    ]))
    cand = [(h,) for h in base]
    for h1 in base[:4]:
        cand.append((h1, max(8, h1 // 2)))
    for h1 in base[-2:]:
        cand.append((h1, max(8, h1 // 2), max(4, h1 // 4)))
    return list(dict.fromkeys(cand))[:max_cand]

def rq4_run_scenario(F_method, y_scen, scenario_tag, method_name, class_names=None):
    """Train ANN-LM via GridSearch on a given (F, y) and return metrics dict."""
    F_scen = F_method if y_scen.shape[0] == F_method.shape[0] else None
    if F_scen is None:
        raise ValueError("y_scen length mismatch with F_method.")

    Xtr, Xte, ytr, yte = train_test_split(
        F_scen, y_scen,
        test_size=RQ4_TEST_FRAC, random_state=RQ4_RANDOM_SEED, stratify=y_scen
    )
    I = Xtr.shape[1]
    O = len(np.unique(ytr))
    cands = rq4_arch_candidates(I, O)

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            random_state=RQ4_RANDOM_SEED,
            max_iter=RQ4_MAX_ITER,
            early_stopping=True,
            n_iter_no_change=RQ4_N_ITER_NO_CHANGE
        ))
    ])
    param_grid = {
        "mlp__hidden_layer_sizes": cands,
        "mlp__alpha": [1e-4, 1e-3, 1e-2],
        "mlp__learning_rate_init": [1e-3, 3e-4],
        "mlp__activation": ["relu", "tanh"],
    }
    cv = StratifiedKFold(n_splits=RQ4_CV_SPLITS, shuffle=True, random_state=RQ4_RANDOM_SEED)
    gs = GridSearchCV(pipe, param_grid, cv=cv, scoring="f1_macro", n_jobs=1, verbose=0)

    t0 = time.perf_counter()
    gs.fit(Xtr, ytr)
    wall = time.perf_counter() - t0

    best = gs.best_estimator_
    pred = best.predict(Xte)
    acc  = accuracy_score(yte, pred)
    prec = precision_score(yte, pred, average="macro", zero_division=0)
    rec  = recall_score(yte, pred, average="macro", zero_division=0)
    f1   = f1_score(yte, pred, average="macro", zero_division=0)

    mlp_best = best.named_steps["mlp"]
    best_hl = mlp_best.hidden_layer_sizes if isinstance(mlp_best.hidden_layer_sizes, tuple)               else (mlp_best.hidden_layer_sizes,)

    return {
        "Scenario": scenario_tag,
        "Method": method_name,
        "n_classes": O,
        "n_train": int(len(Xtr)),
        "n_test": int(len(Xte)),
        "Best_Architecture": str(best_hl),
        "Best_Activation": mlp_best.activation,
        "Best_Alpha": float(mlp_best.alpha),
        "CV_F1_macro": round(float(gs.best_score_), 4),
        "Accuracy": round(float(acc), 4),
        "Precision": round(float(prec), 4),
        "Recall": round(float(rec), 4),
        "F1_macro": round(float(f1), 4),
        "wall_s": round(wall, 2),
        "_gs": gs, "_yte": yte, "_pred": pred, "_class_names": class_names
    }

print("Helper RQ4 siap.")
print(f"RQ4 akan menjalankan Grid Search ANN-LM untuk {len(METHODS_PAPER)} metode x 5 skenario.")
print("⚠️  Ini membutuhkan waktu beberapa menit. Pastikan semua cell di atas sudah dijalankan.")


In [ ]:
# === RQ4: Jalankan 5 Skenario × 2 Metode ===
# CATATAN: cell ini menjalankan 10 GridSearch ANN-LM.
# Waktu estimasi: 5–30 menit tergantung ukuran data & hardware.

rq4_results_raw = []

# Bangun label per skenario
scenarios_def = {}

# S1 – binary (fault-free vs faulty)
y_s1 = build_labels_S1(y_s)
scenarios_def["S1"] = {
    "y": y_s1,
    "keep": np.arange(len(y_s)),
    "class_names": ["fault-free", "faulty"],
    "desc": "Binary: fault-free vs all-fault"
}

# S2 – fault-free vs 4 single faults (5-class)
keep_s2, y_s2 = build_labels_S2(y_s, scenario_names)
scenarios_def["S2"] = {
    "y": y_s2,
    "keep": keep_s2,
    "class_names": ["normal"] + SINGLE_FAULT_NAMES,
    "desc": "5-class: normal + 4 single faults"
}

# S3 – fault-free vs 2-fault combos
keep_s3, y_s3 = build_labels_S3(y_s, scenario_names)
scenarios_def["S3"] = {
    "y": y_s3,
    "keep": keep_s3,
    "class_names": ["normal"] + MULTI2_FAULT_NAMES,
    "desc": "Multi-class: normal + 2-fault combinations"
}

# S4 – single fault identification (4-class, tanpa normal)
keep_s4, y_s4 = build_labels_S4(y_s, scenario_names)
scenarios_def["S4"] = {
    "y": y_s4,
    "keep": keep_s4,
    "class_names": SINGLE_FAULT_NAMES,
    "desc": "4-class: single fault identification"
}

# S5 – all-class (normal + single + multi-fault 2-fault)
keep_s5, y_s5 = build_labels_S5(y_s, scenario_names)
scenarios_def["S5"] = {
    "y": y_s5,
    "keep": keep_s5,
    "class_names": ["normal"] + SINGLE_FAULT_NAMES + MULTI2_FAULT_NAMES,
    "desc": "All-class: normal + single + 2-fault combos"
}

print("Distribusi sampel per skenario:")
for sname, sdef in scenarios_def.items():
    classes, counts = np.unique(sdef["y"], return_counts=True)
    print(f"  {sname}: {sdef['desc']} | n={len(sdef['y'])}, classes={dict(zip(classes.tolist(), counts.tolist()))}")

print()

# Jalankan evaluasi
for sname, sdef in scenarios_def.items():
    y_scen = sdef["y"]
    keep_idx = sdef["keep"]
    cn = sdef["class_names"]

    for method in METHODS_PAPER:
        if method not in F_by_method:
            print(f"SKIP: {method} tidak ada di F_by_method.")
            continue
        F_method_sub = F_by_method[method][keep_idx]

        print(f"▶ {sname} | {method} | n={len(y_scen)} | n_classes={len(np.unique(y_scen))} ...", end=" ", flush=True)
        try:
            row = rq4_run_scenario(F_method_sub, y_scen, sname, method, class_names=cn)
            rq4_results_raw.append(row)
            print(f"F1={row['F1_macro']:.4f} Acc={row['Accuracy']:.4f} [{row['wall_s']:.1f}s]")
        except Exception as e:
            print(f"ERROR: {e}")
            rq4_results_raw.append({
                "Scenario": sname, "Method": method,
                "F1_macro": np.nan, "Accuracy": np.nan,
                "Precision": np.nan, "Recall": np.nan,
                "CV_F1_macro": np.nan, "Best_Architecture": "ERROR",
                "_error": str(e)
            })

print("\n✅ RQ4 selesai.")


In [ ]:
# === RQ4: Tabel Ringkasan Hasil Klasifikasi ===

display_cols = ["Scenario", "Method", "n_classes", "n_train", "n_test",
                "Best_Architecture", "Best_Activation", "CV_F1_macro",
                "Accuracy", "Precision", "Recall", "F1_macro", "wall_s"]

rq4_table = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in rq4_results_raw
])

# Tabel utama paper
print("\n=== RQ4: Hasil Klasifikasi – Lima Skenario (EDM-Fuzzy vs JSD-Fuzzy) ===")
print(rq4_table[[c for c in display_cols if c in rq4_table.columns]].to_string(index=False))

# Export
rq4_table.to_csv("exports/rq4_classification_results.csv", index=False)
print("\n[Saved] exports/rq4_classification_results.csv")


In [ ]:
# === RQ4: Plot Perbandingan F1-Score per Skenario ===
import matplotlib.pyplot as plt
import numpy as np

scenarios_order = ["S1", "S2", "S3", "S4", "S5"]
rq4_pivot = rq4_table.pivot(index="Scenario", columns="Method", values="F1_macro")

x = np.arange(len(scenarios_order))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1-score comparison
ax = axes[0]
for j, method in enumerate(METHODS_PAPER):
    if method not in rq4_pivot.columns:
        continue
    vals = [rq4_pivot.loc[s, method] if s in rq4_pivot.index else np.nan
            for s in scenarios_order]
    ax.bar(x + j*width - width/2, vals, width, label=method, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(scenarios_order)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Macro F1-Score")
ax.set_title("RQ4: F1-Score per Skenario")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Accuracy comparison
rq4_acc_pivot = rq4_table.pivot(index="Scenario", columns="Method", values="Accuracy")
ax2 = axes[1]
for j, method in enumerate(METHODS_PAPER):
    if method not in rq4_acc_pivot.columns:
        continue
    vals = [rq4_acc_pivot.loc[s, method] if s in rq4_acc_pivot.index else np.nan
            for s in scenarios_order]
    ax2.bar(x + j*width - width/2, vals, width, label=method, alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(scenarios_order)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("Accuracy")
ax2.set_title("RQ4: Accuracy per Skenario")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("exports/rq4_performance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] exports/rq4_performance_comparison.png")


In [ ]:
# === RQ4: Confusion Matrix per Skenario per Metode ===
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

for row in rq4_results_raw:
    if "_yte" not in row or "_pred" not in row:
        continue
    sname   = row["Scenario"]
    method  = row["Method"]
    yte     = row["_yte"]
    pred    = row["_pred"]
    cn      = row.get("_class_names", None)
    n_cls   = len(np.unique(yte))

    fig, ax = plt.subplots(figsize=(max(6, n_cls*1.1), max(5, n_cls)))
    cm = confusion_matrix(yte, pred, labels=np.arange(n_cls))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=cn if cn and len(cn) == n_cls else np.arange(n_cls))
    disp.plot(ax=ax, xticks_rotation=40, colorbar=True)
    ax.set_title(f"CM – {sname} | {method}\nF1={row['F1_macro']:.4f} Acc={row['Accuracy']:.4f}")
    plt.tight_layout()
    fname = f"exports/rq4_cm_{sname}_{method.replace('-','_').replace(' ','_')}.png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"[Saved] {fname}")


## RQ5 – Validasi Statistik: Paired t-test F1-score

Menguji apakah perbedaan F1-score antara **JSD-Fuzzy dan EDM-Fuzzy** signifikan secara statistik pada lima skenario klasifikasi (data berpasangan per skenario).

- H₀: μ(F1_JSD) = μ(F1_EDM) → tidak ada perbedaan signifikan
- H₁: μ(F1_JSD) ≠ μ(F1_EDM)
- Signifikan jika **p-value < 0.05**


In [ ]:
# === RQ5: Paired t-test F1-score EDM-Fuzzy vs JSD-Fuzzy ===
from scipy import stats

scenarios_order = ["S1", "S2", "S3", "S4", "S5"]

# Susun F1 berpasangan
f1_edm, f1_jsd = [], []
rq5_rows = []

for sname in scenarios_order:
    edm_row = next((r for r in rq4_results_raw if r["Scenario"]==sname and r["Method"]=="EDM-Fuzzy"), None)
    jsd_row = next((r for r in rq4_results_raw if r["Scenario"]==sname and r["Method"]=="JSD-Fuzzy"), None)

    edm_f1 = edm_row["F1_macro"] if edm_row else np.nan
    jsd_f1 = jsd_row["F1_macro"] if jsd_row else np.nan
    diff   = round(jsd_f1 - edm_f1, 4) if (not np.isnan(edm_f1) and not np.isnan(jsd_f1)) else np.nan

    f1_edm.append(edm_f1)
    f1_jsd.append(jsd_f1)
    rq5_rows.append({
        "Scenario": sname,
        "F1_EDM-Fuzzy": edm_f1,
        "F1_JSD-Fuzzy": jsd_f1,
        "Difference (JSD-EDM)": diff
    })

rq5_table = pd.DataFrame(rq5_rows)

# Paired t-test (hanya pada pasangan yang tidak NaN)
valid_mask = ~(np.isnan(f1_edm) | np.isnan(f1_jsd))
f1_edm_v = np.array(f1_edm)[valid_mask]
f1_jsd_v = np.array(f1_jsd)[valid_mask]

if len(f1_edm_v) >= 2:
    t_stat, p_val = stats.ttest_rel(f1_jsd_v, f1_edm_v)
    mean_diff = (f1_jsd_v - f1_edm_v).mean()
    significant = "Ya (p < 0.05)" if p_val < 0.05 else "Tidak (p ≥ 0.05)"
else:
    t_stat, p_val, mean_diff, significant = np.nan, np.nan, np.nan, "N/A (data tidak cukup)"

print("=== RQ5: Paired t-test F1-score ===")
print(rq5_table.to_string(index=False))
print(f"\nMean F1 EDM-Fuzzy : {np.nanmean(f1_edm):.4f}")
print(f"Mean F1 JSD-Fuzzy : {np.nanmean(f1_jsd):.4f}")
print(f"Mean Difference   : {mean_diff:.4f} (JSD - EDM)")
print(f"t-statistic       : {t_stat:.4f}")
print(f"p-value           : {p_val:.4f}")
print(f"Signifikan        : {significant}")

rq5_table["t_stat"] = t_stat
rq5_table["p_value"] = p_val
rq5_table["Signifikan"] = significant
rq5_table.to_csv("exports/rq5_paired_ttest.csv", index=False)
print("\n[Saved] exports/rq5_paired_ttest.csv")


In [ ]:
# === RQ5: Visualisasi F1-Score Berpasangan ===
import matplotlib.pyplot as plt
import numpy as np

scenarios_order_plot = ["S1", "S2", "S3", "S4", "S5"]
f1_edm_plot = [rq5_table.loc[rq5_table["Scenario"]==s, "F1_EDM-Fuzzy"].values[0]
               if s in rq5_table["Scenario"].values else np.nan for s in scenarios_order_plot]
f1_jsd_plot = [rq5_table.loc[rq5_table["Scenario"]==s, "F1_JSD-Fuzzy"].values[0]
               if s in rq5_table["Scenario"].values else np.nan for s in scenarios_order_plot]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(scenarios_order_plot))
ax.plot(x, f1_edm_plot, "o--", label="EDM-Fuzzy", linewidth=2, markersize=8)
ax.plot(x, f1_jsd_plot, "s-", label="JSD-Fuzzy", linewidth=2, markersize=8)
ax.set_xticks(x)
ax.set_xticklabels(scenarios_order_plot)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Macro F1-Score")
ax.set_xlabel("Skenario Klasifikasi")
ax.set_title(f"RQ5: F1-Score EDM-Fuzzy vs JSD-Fuzzy (p={p_val:.4f}, {significant})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("exports/rq5_f1_paired_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] exports/rq5_f1_paired_comparison.png")


## Ringkasan Seluruh Hasil (Paper Summary Table)

Tabel gabungan untuk kebutuhan paper: RQ1–RQ5 dalam satu view.


In [ ]:
# === Ringkasan Akhir – Paper Summary ===

print("=" * 70)
print("RINGKASAN PAPER: JSD-Fuzzy vs EDM-Fuzzy")
print("=" * 70)

print("\n[RQ1] Feature Matrix Validation:")
if "rq1_table" in dir() or "rq1_table" in globals():
    print(rq1_table.to_string(index=False))

print("\n[RQ2] Mean CV per Scale (averaged across all classes):")
if "rq2_by_scale" in globals():
    print(rq2_by_scale.round(4).to_string())
    if "CV_Reduction_%" in rq2_by_scale.columns:
        mean_reduc = rq2_by_scale["CV_Reduction_%"].mean()
        n_better = (rq2_by_scale["CV_Reduction_%"] > 0).sum()
        print(f"  → JSD-Fuzzy lebih stabil pada {n_better}/{len(rq2_by_scale)} scale, rata-rata CV reduction = {mean_reduc:.2f}%")

print("\n[RQ4] Klasifikasi Lima Skenario:")
if "rq4_table" in globals():
    print(rq4_table[["Scenario","Method","Accuracy","Precision","Recall","F1_macro"]].to_string(index=False))

print("\n[RQ5] Paired t-test:")
if "rq5_table" in globals():
    print(rq5_table[["Scenario","F1_EDM-Fuzzy","F1_JSD-Fuzzy","Difference (JSD-EDM)","p_value","Signifikan"]].to_string(index=False))

# Export combined summary
summary_frames = []
if "rq1_table" in globals():
    summary_frames.append(("rq1_table", rq1_table))
if "rq4_table" in globals():
    summary_frames.append(("rq4_table", rq4_table[["Scenario","Method","Accuracy","Precision","Recall","F1_macro"]]))
if "rq5_table" in globals():
    summary_frames.append(("rq5_table", rq5_table))

with pd.ExcelWriter("exports/paper_summary_tables.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in summary_frames:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    if "rq2_by_scale" in globals():
        rq2_by_scale.to_excel(writer, sheet_name="rq2_cv_by_scale")
    if "rq3_table" in globals():
        rq3_table.to_excel(writer, sheet_name="rq3_separability", index=False)

print("\n[Saved] exports/paper_summary_tables.xlsx")


In [ ]:
# Export all key tables (CSV + Parquet) with clickable links
export_df(feature_metrics_table, "feature_metrics_table", index=True)
export_df(train_metrics_table, "train_metrics_table", index=True)
export_df(combined_metrics_table, "combined_metrics_table", index=True)
if "cv_table" in globals():
    export_df(cv_table, "cv_table", index=False)
if "mean_entropy_table" in globals():
    export_df(mean_entropy_table, "mean_entropy_table", index=True)
if "exp" in globals():
    export_df(exp, "experiment_S_table", index=False)
if "scenario_table" in globals():
    export_df(scenario_table, "scenario_ann_tradeoff_table", index=False)


### Notes (what to adjust next run)

**Main switches**
- `FAST_MODE`: fastest end-to-end run (smaller sampling + lighter ANN search). Set `False` for more thorough accuracy runs.
- `RUN_ALL_METHODS`: compute/evaluate all entropy methods in `METHOD_LIST`.
- `DEFAULT_METHOD`: which method is used for the “single-method” plots/reports when you only want one highlighted.

**Speed knobs**
- `MAX_PER_CLASS`, `WIN`, `STRIDE`, `S`, `n_ref`: biggest impact on runtime (entropy dominates).
- If you re-run often, keep `CACHE_FEATURES=True` so features load from `cache/` instead of recomputing.

**Quality knobs**
- Increase `MAX_PER_CLASS`, `S`, and `n_ref` for better estimates (slower).
- Turn on `USE_TIME_SPLIT=True` to reduce leakage when windows overlap (may lower apparent accuracy but is more honest).
- Expand `param_grid` (hidden sizes / alpha / lr / activation) for a stronger ANN search (slower).

**Export**
All important tables are saved to `exports/` as CSV (and Parquet when available) with clickable links in the last cell.
